# Task 2: Citation Mapping - BERT Training (LR Ablation)

**Model:** bert-base-uncased  
**Phase:** Phase 2 — Context Mode Ablation  
**Run:** task2-m4pro-bert-lr3e5-bs32-ep3-window2

In [1]:
import transformers, datasets, accelerate
print(f"✅ transformers: {transformers.__version__}")
print(f"✅ datasets: {datasets.__version__}")
print(f"✅ accelerate: {accelerate.__version__}")

/Users/tathiyennhi/Documents/Works/automatic-citation-checking/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


✅ transformers: 4.57.6
✅ datasets: 4.5.0
✅ accelerate: 1.10.1


In [2]:
import wandb
import os
from dotenv import load_dotenv

load_dotenv()
key = os.environ.get("WANDB_API_KEY")

if key:
    wandb.login(key=key, relogin=True)
    print("✅ Wandb logged in")
else:
    os.environ["WANDB_MODE"] = "disabled"
    print("⚠️ Wandb disabled — WANDB_API_KEY not found in .env")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/tathiyennhi/.netrc
wandb: Currently logged in as: tathiyennhi (tathiyennhi-hcmus) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ Wandb logged in


In [3]:
import math

# ═══════════════════════════════════════════════════════════
# ⚙️ CONFIG — Phase 1: Learning Rate Ablation
# ═══════════════════════════════════════════════════════════
LEARNING_RATE               = 3e-5
CONTEXT_MODE                = "window_2"
NEG_RATIO                   = 3
NUM_EPOCHS                  = 3
PER_DEVICE_TRAIN_BATCH_SIZE = 8
PER_DEVICE_EVAL_BATCH_SIZE  = 8
GRADIENT_ACCUMULATION_STEPS = 4       # effective batch = 32
WARMUP_RATIO                = 0.1
WEIGHT_DECAY                = 0.01
SEED                        = 42
MODEL_NAME                  = "bert-base-uncased"
MAX_LENGTH                  = 512

EFFECTIVE_BATCH_SIZE = PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
RUN_NAME = f"task2-m4pro-bert-lr3e5-bs{EFFECTIVE_BATCH_SIZE}-ep{NUM_EPOCHS}-window_2"

print(f"✅ Config loaded | LR={LEARNING_RATE} | eff_batch={EFFECTIVE_BATCH_SIZE} | epochs={NUM_EPOCHS} | context={CONTEXT_MODE}")

✅ Config loaded | LR=3e-05 | eff_batch=32 | epochs=3 | context=window_2


In [4]:
import os
from pathlib import Path

DATA_ROOT = Path().resolve() / "data_outputs/task2"

train_path = os.path.join(DATA_ROOT, "train")
val_path = os.path.join(DATA_ROOT, "val")

train_count = len([f for f in os.listdir(train_path) if f.endswith('.label')])
val_count = len([f for f in os.listdir(val_path) if f.endswith('.label')])

print(f"✅ Train: {train_count:,} files")
print(f"✅ Val: {val_count:,} files")

✅ Train: 55,556 files
✅ Val: 3,000 files


In [5]:
import json
import re
import random
from pathlib import Path
from datasets import Dataset

random.seed(SEED)


def get_context(text, citation_id, mode='full'):
    if mode == 'full':
        return text

    window = int(mode.split('_')[1])
    sentences = re.split(r'(?<=[.!?])\s+', text)

    target_idx = -1
    for i, sent in enumerate(sentences):
        if citation_id in sent:
            target_idx = i
            break

    if target_idx == -1:
        return text

    start = max(0, target_idx - window)
    end = min(len(sentences), target_idx + window + 1)
    return ' '.join(sentences[start:end])


def load_task2_data(data_dir, max_files=None, neg_ratio=3, context_mode='full'):
    data_path = Path(data_dir)
    label_files = sorted(data_path.glob('*.label'))

    if max_files:
        label_files = label_files[:max_files]

    total_files = len(label_files)
    print(f'📊 Loading {total_files:,} files | Mode: {context_mode}')

    examples = []
    skipped = 0
    stats = {'positive': 0, 'negative': 0}

    for file_idx, label_file in enumerate(label_files):
        if (file_idx + 1) % 1000 == 0:
            print(f'⏳ {file_idx+1:,}/{total_files:,} | Examples: {len(examples):,}')

        in_file = label_file.with_suffix('.in')

        try:
            with open(in_file) as f:
                in_data = json.load(f)
            with open(label_file) as f:
                label_data = json.load(f)
        except:
            skipped += 1
            continue

        text = in_data.get('text', '')
        if not text:
            skipped += 1
            continue

        candidates = in_data.get('citation_candidates', [])
        bib_entries = in_data.get('bib_entries', {})
        correct_citation = label_data.get('correct_citation', {})

        if not correct_citation or not candidates or not bib_entries:
            skipped += 1
            continue

        for citation_id, correct_paper_id in correct_citation.items():
            context = get_context(text, citation_id, mode=context_mode)

            if correct_paper_id in bib_entries:
                paper = bib_entries[correct_paper_id]
                paper_text = f"{paper.get('title', '')}. {paper.get('abstract', '')}"
                examples.append({'text_a': context, 'text_b': paper_text, 'label': 1})
                stats['positive'] += 1

            neg_candidates = [c for c in candidates if c != correct_paper_id and c in bib_entries]
            neg_sample = random.sample(neg_candidates, min(neg_ratio, len(neg_candidates)))

            for neg_paper_id in neg_sample:
                paper = bib_entries[neg_paper_id]
                paper_text = f"{paper.get('title', '')}. {paper.get('abstract', '')}"
                examples.append({'text_a': context, 'text_b': paper_text, 'label': 0})
                stats['negative'] += 1

    print(f'\n✅ {len(examples):,} examples | Pos: {stats["positive"]:,} | Neg: {stats["negative"]:,} | Skip: {skipped}')
    return examples


print('=' * 60)
print(f'CONTEXT MODE: {CONTEXT_MODE} | NEG_RATIO: {NEG_RATIO}')
print('=' * 60)

train_examples = load_task2_data(train_path, neg_ratio=NEG_RATIO, context_mode=CONTEXT_MODE)
val_examples = load_task2_data(val_path, neg_ratio=NEG_RATIO, context_mode=CONTEXT_MODE)

train_dataset = Dataset.from_list(train_examples)
val_dataset = Dataset.from_list(val_examples)

print(f'\n✅ Train: {len(train_dataset):,} | Val: {len(val_dataset):,}')

CONTEXT MODE: window_2 | NEG_RATIO: 3
📊 Loading 55,556 files | Mode: window_2
⏳ 1,000/55,556 | Examples: 9,354
⏳ 2,000/55,556 | Examples: 19,564
⏳ 3,000/55,556 | Examples: 28,466
⏳ 4,000/55,556 | Examples: 37,282
⏳ 5,000/55,556 | Examples: 46,464
⏳ 6,000/55,556 | Examples: 55,701
⏳ 7,000/55,556 | Examples: 64,876
⏳ 8,000/55,556 | Examples: 74,497
⏳ 9,000/55,556 | Examples: 83,561
⏳ 10,000/55,556 | Examples: 92,538
⏳ 11,000/55,556 | Examples: 101,509
⏳ 12,000/55,556 | Examples: 110,616
⏳ 13,000/55,556 | Examples: 120,131
⏳ 14,000/55,556 | Examples: 129,434
⏳ 15,000/55,556 | Examples: 137,777
⏳ 16,000/55,556 | Examples: 146,906
⏳ 17,000/55,556 | Examples: 156,140
⏳ 18,000/55,556 | Examples: 164,600
⏳ 19,000/55,556 | Examples: 173,365
⏳ 20,000/55,556 | Examples: 182,450
⏳ 21,000/55,556 | Examples: 191,771
⏳ 22,000/55,556 | Examples: 201,003
⏳ 23,000/55,556 | Examples: 210,029
⏳ 24,000/55,556 | Examples: 218,620
⏳ 25,000/55,556 | Examples: 227,619
⏳ 26,000/55,556 | Examples: 236,468
⏳ 27,0

In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"✅ Tokenizer: {MODEL_NAME}")


def tokenize_function(examples):
    return tokenizer(
        examples["text_a"],
        examples["text_b"],
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length",
    )


print("Tokenizing train...")
train_tokenized = train_dataset.map(tokenize_function, batched=True, remove_columns=["text_a", "text_b"])

print("Tokenizing val...")
val_tokenized = val_dataset.map(tokenize_function, batched=True, remove_columns=["text_a", "text_b"])

print(f"\n✅ Train: {len(train_tokenized):,} | Val: {len(val_tokenized):,}")

✅ Tokenizer: bert-base-uncased
Tokenizing train...


Map:   0%|          | 0/511596 [00:00<?, ? examples/s]

Tokenizing val...


Map:   0%|          | 0/27982 [00:00<?, ? examples/s]


✅ Train: 511,596 | Val: 27,982


In [7]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
print(f"✅ Model: {MODEL_NAME} (num_labels=2)")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Model: bert-base-uncased (num_labels=2)


In [8]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", pos_label=1)
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}


print("✅ Metrics defined")

✅ Metrics defined


In [9]:
from transformers import TrainingArguments, Trainer

WANDB_PROJECT  = "task2-citation-mapping"
CHECKPOINT_DIR = f"./outputs/checkpoints/{RUN_NAME}"
SAVE_DIR       = f"./outputs/models/{RUN_NAME}"
CONFIG_DIR     = f"./outputs/configs/{RUN_NAME}.json"

try:
    if wandb.run is None:
        wandb.init(
            project=WANDB_PROJECT,
            name=RUN_NAME,
            config={
                "model":                       MODEL_NAME,
                "learning_rate":               LEARNING_RATE,
                "num_train_epochs":            NUM_EPOCHS,
                "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
                "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
                "effective_batch_size":        EFFECTIVE_BATCH_SIZE,
                "warmup_ratio":                WARMUP_RATIO,
                "weight_decay":                WEIGHT_DECAY,
                "context_mode":                CONTEXT_MODE,
                "neg_ratio":                   NEG_RATIO,
                "seed":                        SEED,
                "max_length":                  MAX_LENGTH,
            },
            resume="allow",
        )
    report_to = "wandb"
    print("✅ Wandb initialized")
except Exception:
    report_to = "none"
    print("⚠️ Wandb not available")


training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_dir="./outputs/logs",
    logging_steps=50,
    report_to=report_to,
    fp16=False,
    bf16=True,
    dataloader_num_workers=0,
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics,
)

print(
    f"\n✅ Trainer ready"
    f"\n   LR={training_args.learning_rate}"
    f"\n   epochs={training_args.num_train_epochs}"
    f"\n   per_device_batch={training_args.per_device_train_batch_size}"
    f"\n   grad_acc={training_args.gradient_accumulation_steps}"
    f"\n   effective_batch={EFFECTIVE_BATCH_SIZE}"
    f"\n   warmup_ratio={training_args.warmup_ratio}"
)

✅ Wandb initialized

✅ Trainer ready
   LR=3e-05
   epochs=3
   per_device_batch=8
   grad_acc=4
   effective_batch=32
   warmup_ratio=0.1


In [10]:
print("=" * 60)
print(f"🚀 TRAINING | {RUN_NAME}")
print("=" * 60)

trainer.train()

print("\n✅ Training complete!")

🚀 TRAINING | task2-m4pro-bert-lr3e5-bs32-ep3-window_2


/Users/tathiyennhi/Documents/Works/automatic-citation-checking/venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.432600,0.428044,0.799764,0.703531,0.362482,0.478451
2,0.415900,0.421972,0.807698,0.701581,0.419464,0.525024
3,0.353300,0.443040,0.806447,0.656566,0.495063,0.564490


/Users/tathiyennhi/Documents/Works/automatic-citation-checking/venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/tathiyennhi/Documents/Works/automatic-citation-checking/venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)



✅ Training complete!


In [11]:
print("📊 VALIDATION RESULTS")
print("=" * 60)

eval_results = trainer.evaluate()

for key, value in eval_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

print("=" * 60)
print(f"\n✅ Accuracy:  {eval_results.get('eval_accuracy', 0):.2%}")
print(f"✅ Precision: {eval_results.get('eval_precision', 0):.2%}")
print(f"✅ Recall:    {eval_results.get('eval_recall', 0):.2%}")
print(f"✅ F1:        {eval_results.get('eval_f1', 0):.2%}")

📊 VALIDATION RESULTS


/Users/tathiyennhi/Documents/Works/automatic-citation-checking/venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


eval_loss: 0.4430
eval_accuracy: 0.8064
eval_precision: 0.6566
eval_recall: 0.4951
eval_f1: 0.5645
eval_runtime: 698.5952
eval_samples_per_second: 40.0550
eval_steps_per_second: 5.0070
epoch: 3.0000

✅ Accuracy:  80.64%
✅ Precision: 65.66%
✅ Recall:    49.51%
✅ F1:        56.45%


In [12]:
import os
import json
import csv
import shutil

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(os.path.dirname(CONFIG_DIR), exist_ok=True)
os.makedirs("./outputs/results", exist_ok=True)

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
shutil.make_archive(SAVE_DIR, 'zip', SAVE_DIR)
print(f"✅ Model saved: {SAVE_DIR}")

# Save config
config_data = {
    "run_name":                    RUN_NAME,
    "model":                       MODEL_NAME,
    "learning_rate":               LEARNING_RATE,
    "num_train_epochs":            NUM_EPOCHS,
    "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "effective_batch_size":        EFFECTIVE_BATCH_SIZE,
    "warmup_ratio":                WARMUP_RATIO,
    "weight_decay":                WEIGHT_DECAY,
    "context_mode":                CONTEXT_MODE,
    "neg_ratio":                   NEG_RATIO,
    "seed":                        SEED,
    "max_length":                  MAX_LENGTH,
    "eval_f1":                     eval_results.get("eval_f1", 0),
    "eval_loss":                   eval_results.get("eval_loss", 0),
    "eval_accuracy":               eval_results.get("eval_accuracy", 0),
    "eval_precision":              eval_results.get("eval_precision", 0),
    "eval_recall":                 eval_results.get("eval_recall", 0),
}
with open(CONFIG_DIR, "w") as f:
    json.dump(config_data, f, indent=2)
print(f"✅ Config saved: {CONFIG_DIR}")

# Append to ablation summary CSV
summary_path = "./outputs/results/ablation_summary.csv"
write_header = not os.path.exists(summary_path)
with open(summary_path, "a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=config_data.keys())
    if write_header:
        writer.writeheader()
    writer.writerow(config_data)
print(f"✅ Appended to: {summary_path}")

try:
    wandb.finish()
except:
    pass

print(f"\n🎉 DONE | {RUN_NAME}")
print(f"   F1:       {eval_results.get('eval_f1', 0):.4f}")
print(f"   Eval loss:{eval_results.get('eval_loss', 0):.4f}")

✅ Model saved: ./outputs/models/task2-m4pro-bert-lr3e5-bs32-ep3-window_2
✅ Config saved: ./outputs/configs/task2-m4pro-bert-lr3e5-bs32-ep3-window_2.json
✅ Appended to: ./outputs/results/ablation_summary.csv


eval/accuracy,▁█▇▇
eval/f1,▁▅██
eval/loss,▃▁██
eval/precision,██▁▁
eval/recall,▁▄██
eval/runtime,█▁██
eval/samples_per_second,▁█▁▁
eval/steps_per_second,▁█▁▁
train/epoch,▁▁▁▁▁▂▂▃▃▃▃▃▃▄▄▄▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇██████
train/global_step,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▅▅▆▇▇▇▇▇▇███
+3,...



🎉 DONE | task2-m4pro-bert-lr3e5-bs32-ep3-window_2
   F1:       0.5645
   Eval loss:0.4430
